## Data Loading

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from sklearn import preprocessing
import numpy as np
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import BertTokenizer, BertModel
import torch
import torch.nn as nn

# load_aokvqa.py
import os
import json

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_aokvqa(aokvqa_dir, split, version='v1p0'):
    assert split in ['train', 'val', 'test', 'test_w_ans']
    dataset = json.load(open(
        os.path.join(aokvqa_dir, f"aokvqa_{version}_{split}.json")
    ))
    return dataset

def get_coco_path(split, image_id, coco_dir):
    return os.path.join(coco_dir, f"{split}2017", f"{image_id:012}.jpg")

AOKVQA_DIR="datasets/aokvqa/"
COCO_DIR="datasets/coco2017/"
aokvqa_dir = f"./aokvqa/{AOKVQA_DIR}"
coco_dir = f"./aokvqa/{COCO_DIR}"

/opt/conda/envs/r3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
def subsample(dataset, n_samples=None, frac=None, random_seed=42):
    random.seed(random_seed)
    if n_samples:
        return random.sample(dataset, n_samples)
    elif frac:
        sample_size = int(len(dataset) * frac)
        return random.sample(dataset, sample_size)
    return dataset

In [3]:
USE_SUBSET_DATA = False 
train_dataset = load_aokvqa(aokvqa_dir, 'train')  
val_dataset = load_aokvqa(aokvqa_dir, 'val')
test_dataset = load_aokvqa(aokvqa_dir, 'test')
print(f"Full Train aokvqa: {len(train_dataset)}")
print(f"Full Val aokvqa: {len(val_dataset)}")
print(f"Full Test aokvqa: {len(test_dataset)}")

if USE_SUBSET_DATA:
    train_dataset = subsample(train_dataset, frac=0.2)
    val_dataset = subsample(val_dataset, frac=0.2)
    test_dataset = subsample(test_dataset, frac=0.2)
    print(f"Used Train aokvqa: {len(train_dataset)}")
    print(f"Used Val aokvqa: {len(val_dataset)}")
    print(f"Used Test aokvqa: {len(test_dataset)}")

Full Train aokvqa: 17056
Full Val aokvqa: 1145
Full Test aokvqa: 6702


## Data Preparation

Fields Considered:

- Question
- Choices
- Correct answer
- Correct Choice Indice
- Rationale
- Direct answer

In [4]:
qa_data = []
for sample in val_dataset:
    question_id = sample["question_id"]
    image_id = sample["image_id"]
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]
    qa_data.append({
        "question_id": question_id,
        "image_id": image_id,
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })
qa_df = pd.DataFrame(qa_data)

In [10]:
qa_df.head()


,question_id,image_id,question,choices,correct_answer,correct_choice_idx,rationale,direct_answer,clip_baseline_prediction,clip_baseline_confidence
0,22jbM6gDxdaMaunuzgrsBB,461751,What is in the motorcyclist's mouth?,"[toothpick, food, popsicle stick, cigarette]",cigarette,3,He's smoking while riding. The motorcyclist ha...,cigarette cigarette cigarette cigarette cigare...,cigarette,0.251709
1,2Aq5RiEn7eyfWjEbpuYT2o,377368,Which number birthday is probably being celebr...,"[one, ten, nine, thirty]",thirty,3,There is a birthday cake on the table with the...,thirty 30th thirty thirty thirty 30th thirty t...,thirty,0.253418
2,2Br4bJfKY7SQM9DECrqqeG,563603,What best describes the pool of water?,"[frozen, fresh, dirty, boiling]",dirty,2,The pool is dark brown. It it brown and surrou...,muddy dirty murky water muddy pond pond wateri...,fresh,0.252441
3,2C8riXpRLX3CyM5jDz23m7,329542,What is the white substance on top of the cupc...,"[butter, mayo, ice cream, icing]",icing,3,This is frosting used to decorate and add more...,icing whipped cream icing frosting icing frost...,butter,0.251709
4,2DQex53EkNGH2cfo3WPuPn,182202,What type of device is sitting next to the lap...,"[mouse, mobile phone, pen, keyboard]",mobile phone,1,It has the name of it on the top The device ha...,cell phone vodafone phone phone phone phone ce...,mouse,0.251709


## Multimodal

### CLIP

In [6]:
def get_clip_prediction(img_path, choices, question):
    # Load and preprocess the image
    image = Image.open(img_path).convert("RGB")
    image_input = preprocess(image).unsqueeze(0).to(device)
    
    # Tokenize the list of candidate text choices
    # text_input = clip.tokenize(choices).to(device)
    text_input = torch.cat([
        clip.tokenize(f"{question} Answer: {c}") for c in choices
    ]).to(device)
    # print(str(text_input))
    
    with torch.no_grad():
        # Compute image and text features
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_input)
        
        # Normalize features to unit length (recommended for cosine similarity)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Compute similarity between the image and each text choice
        similarity = (image_features @ text_features.T).squeeze(0)

        confidence_scores = torch.softmax(similarity, dim=0).cpu().numpy()
    
    # Return the choice with the highest similarity score
    best_idx = similarity.argmax().item()
    best_answer = choices[best_idx]
    best_confidence = confidence_scores[best_idx]
    
    return best_answer, best_confidence

In [7]:
import torch
import clip
from PIL import Image
from tqdm import tqdm

# Set up device and load the CLIP model along with its preprocessing pipeline.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

# Process each sample in qa_data and compute predictions
predictions = []
condifence_scores = []
for sample in tqdm(qa_data, desc="Processing Images"):
    image_id = sample['image_id']
    img_path = get_coco_path('val', image_id, coco_dir)
    choices  = sample['choices'] 
    prediction, confidence = get_clip_prediction(img_path, choices, sample['question'])
    predictions.append(prediction)
    condifence_scores.append(confidence)

qa_df['clip_baseline_prediction'] = predictions
qa_df['clip_baseline_confidence'] = condifence_scores


Processing Images: 100%|██████████| 1145/1145 [02:24<00:00,  7.90it/s]


In [11]:
correctness = (qa_df['clip_baseline_prediction'] == qa_df['correct_answer'])
accuracy = correctness.mean()
calibration_error = (qa_df['clip_baseline_confidence'] - correctness).abs().mean()
print(f"CLIP Baseline Accuracy: {accuracy:.2%}")
print(f"CLIP Baseline Calibration Error: {calibration_error:.2%}")

CLIP Baseline Accuracy: 54.93%
CLIP Baseline Calibration Error: 52.29%


In [8]:
qa_df.to_pickle("CLIP_result.pkl")

# VisualBERT
## Multiple Choice

In [78]:
def extract_visual_features(img_path, max_regions=36):
    """Extract region features using Faster R-CNN's ROI heads and upsample to 2048 dimensions."""
    img = Image.open(img_path).convert("RGB")
    img_array = np.array(img)
    
    # Create a linear layer to upsample features from 1028 to 2048
    upsample_layer = nn.Linear(1028, 2048).to(device)
    
    with torch.no_grad():
        # Prepare input for Detectron2
        height, width = img_array.shape[:2]
        inputs = [{"image": torch.as_tensor(img_array).permute(2, 0, 1).to(device), 
                   "height": height, 
                   "width": width}]
        
        # Run detector
        images = predictor.model.preprocess_image(inputs)
        features = predictor.model.backbone(images.tensor)
        proposals, _ = predictor.model.proposal_generator(images, features)

        # Get ROI features
        box_features = predictor.model.roi_heads.box_pooler(
            [features[f] for f in predictor.model.roi_heads.box_in_features],
            [p.proposal_boxes for p in proposals]
        )
        roi_features = predictor.model.roi_heads.box_head(box_features)

        # Get top-k proposals by objectness score
        scores = proposals[0].objectness_logits.sigmoid().cpu().numpy()
        top_k = min(max_regions, len(scores))
        indices = np.argsort(scores)[-top_k:]
        
        # Extract top-k boxes & features
        boxes = proposals[0].proposal_boxes.tensor[indices].cpu().numpy()
        features = roi_features[indices].cpu().numpy()  # Shape: [N, feature_dim]
        
        # Normalize bounding boxes
        boxes[:, 2] -= boxes[:, 0]  # x2 -> w
        boxes[:, 3] -= boxes[:, 1]  # y2 -> h
        boxes[:, [0, 2]] /= width    # Normalize x, w
        boxes[:, [1, 3]] /= height   # Normalize y, h
        
        # Combine features and boxes
        visual_embeds = np.concatenate([features, boxes], axis=1)
        visual_embeds = torch.from_numpy(visual_embeds).float().unsqueeze(0).to(device)

        # Upsample the visual embeddings from 1028 to 2048
        visual_embeds = upsample_layer(visual_embeds.squeeze(0))  # Remove batch dim before applying Linear layer
        visual_embeds = visual_embeds.unsqueeze(0)  # Add back batch dim
    
    return visual_embeds

In [157]:
import torch
import numpy as np
from PIL import Image

def visualbert_answer(img_path, question, choices=None):
    """Run VisualBERT for multiple-choice or direct answering with confidence scores."""
    visual_embeds = extract_visual_features(img_path)
    visual_attention_mask = torch.ones(visual_embeds.shape[:2]).to(device)
    visual_token_type_ids = torch.ones(visual_embeds.shape[:2]).to(device)

    # Multiple-choice processing
    input_ids, attention_masks, token_type_ids = [], [], []
    print(choices)
    for choice in choices:
        text = f"{question} [SEP] {choice}"
        inputs = tokenizer(
            text,
            return_tensors="pt",
            padding="max_length",
            max_length=128,
            truncation=True
        )
        print(len(inputs["input_ids"]))
        input_ids.append(inputs["input_ids"])
        attention_masks.append(inputs["attention_mask"])
        token_type_ids.append(inputs["token_type_ids"])
    # print(input_ids[0])
    input_ids = torch.cat(input_ids, dim=0).to(device)
    print(input_ids.shape)
    attention_mask = torch.cat(attention_masks, dim=0).to(device)
    token_type_ids = torch.cat(token_type_ids, dim=0).to(device)
    
    print(visual_embeds.shape)
    print(visual_attention_mask.shape)
    print(visual_token_type_ids.shape)
    with torch.no_grad():
        batch_size = input_ids.shape[0]  # Number of choices
        visual_embeds = visual_embeds.expand(batch_size, -1, -1)  # Expand visual features
        visual_attention_mask = visual_attention_mask.expand(batch_size, -1)
        visual_token_type_ids = visual_token_type_ids.expand(batch_size, -1)

        print(input_ids.dtype)
        print(attention_mask.dtype)
        print(token_type_ids.dtype)
        print(visual_embeds.dtype)
        print(visual_attention_mask.dtype)
        print(visual_token_type_ids.dtype)
        print(input_ids.shape)
        print(attention_mask.shape)
        print(token_type_ids.shape)
        print(visual_embeds.shape)
        print(visual_attention_mask.shape)
        print(visual_token_type_ids.shape)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds.float(),
            visual_attention_mask=visual_attention_mask.long(),
            visual_token_type_ids=visual_token_type_ids.long(),
        )
        logits = outputs.logits
        choice_confidence_scores = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    
    best_idx = logits.argmax().item()
    # return choices[best_idx], float(choice_confidence_scores[best_idx])

    # Direct answering
    inputs = tokenizer(
        question,
        return_tensors="pt",
        padding="max_length",
        max_length=128,
        truncation=True
    )
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    token_type_ids = inputs["token_type_ids"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds,
            visual_attention_mask=visual_attention_mask,
            visual_token_type_ids=visual_token_type_ids,
            output_hidden_states=True,  # Get token probabilities
            return_dict=True
        )
        logits = outputs.logits  # Get token logits
        probs = torch.softmax(logits, dim=-1)  # Convert logits to probabilities

        # Generate answer
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            visual_embeds=visual_embeds,
            visual_attention_mask=visual_attention_mask,
            visual_token_type_ids=visual_token_type_ids,
            max_length=20
        )
        direct_answer = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        # Compute confidence as mean probability of generated tokens
        generated_probs = probs[0, torch.arange(len(generated_ids[0])), generated_ids[0]]
        confidence_score = float(torch.mean(generated_probs).item())

        return choices[best_idx], float(choice_confidence_scores[best_idx]), direct_answer, confidence_score


In [133]:
    # print(input_ids.shape)
    # print(attention_mask.shape)
    # print(token_type_ids.shape)
    # print(visual_embeds.shape)
    # print(visual_attention_mask.shape)
    # print(visual_token_type_ids.shape)

In [134]:
import torch
from PIL import Image
from transformers import VisualBertForMultipleChoice, BertTokenizer
from torchvision import transforms

model = VisualBertForMultipleChoice.from_pretrained("uclanlp/visualbert-vqa", ignore_mismatched_sizes=True)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # VisualBERT uses BERT's tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

import torch
import numpy as np
from PIL import Image
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg

# Initialize Faster R-CNN detector
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Set confidence threshold
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml")  # Use COCO weights
cfg.MODEL.ROI_HEADS.OUTPUT_FEATURES = True
predictor = DefaultPredictor(cfg)

Some weights of VisualBertForMultipleChoice were not initialized from the model checkpoint at uclanlp/visualbert-vqa and are newly initialized because the shapes did not match:
- cls.weight: found shape torch.Size([3129, 768]) in the checkpoint and torch.Size([1, 768]) in the model instantiated
- cls.bias: found shape torch.Size([3129]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [159]:
# Process each sample in qa_data and compute predictions
choices_pred = []
choice_confidences = []
direct_answers = []
direct_confidences = []
for sample in tqdm(qa_data, desc="Processing Images"):
    image_id = sample['image_id']
    img_path = get_coco_path('val', image_id, coco_dir)
    choices  = sample['choices'] 
    choice, confidence_choice, direct, confident_direct = visualbert_answer(img_path, sample['question'], choices)
    choices_pred.append(choice)
    choice_confidences.append(confidence_choice)
    direct_answers.append(direct)
    direct_confidences.append(confident_direct)


qa_df['VisualBERT_multiple_choice_prediction'] = choices_pred
qa_df['VisualBERT_multiple_choice_prediction_confidence'] = choice_confidences
qa_df['VisualBERT_direct_answers_prediction'] = direct_answers
qa_df['VisualBERT_direct_answers_prediction_confidence'] = direct_confidences

Processing Images:   0%|          | 0/1145 [00:00<?, ?it/s]

['toothpick', 'food', 'popsicle stick', 'cigarette']
1
1
1
1
torch.Size([4, 128])
torch.Size([1, 36, 2048])
torch.Size([1, 36])
torch.Size([1, 36])
torch.int64
torch.int64
torch.int64
torch.float32
torch.float32
torch.float32
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 36, 2048])
torch.Size([4, 36])
torch.Size([4, 36])


RuntimeError: shape '[-1, 128]' is invalid for input of size 4

# NEW